In [1]:
from pathlib import Path
import pandas as pd 
import glob
import numpy as np
from tqdm.auto import tqdm
tqdm.pandas()
import xgboost as xgb
from scipy.optimize import minimize_scalar
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from scipy.stats import pearsonr, spearmanr

root = Path('/data/data/malpolon/xgb/')
inputs_path = Path('/marbec-data/RLS-Australia/malpolon/inputs/australia/')
output_path = Path('/marbec-data/RLS-Australia/malpolon/outputs/')

In [2]:
fulldf = pd.read_csv(inputs_path / 'biomass_lognorm.csv', index_col='survey_id',
                     dtype = {22:str, 24:str, 25:str})
species = list(fulldf.columns[-818:-1])
groundtruth = fulldf[species]

## Preparation

In [6]:
## Calculate predictors
 
def get_compound_values(row):

    survey_id = row.name

    filename = Path(inputs_path) / "env" / (str(survey_id) + '.npy')
    x = np.load(filename).astype(np.float32)
    env = np.transpose(x, (2, 0, 1))

    filename = Path(inputs_path) / "hum" / (str(survey_id) + '.npy')
    x = np.load(filename).astype(np.float32)
    hum = np.transpose(x, (2, 0, 1))

    envhum = np.concatenate([env, hum], axis=0)

    center_values = 0.25 * (envhum[:, 16, 16] + envhum[:, 15, 16] + envhum[:, 16, 15] + envhum[:, 15, 15])
    means = np.mean(envhum, axis=(1,2))
    deviations = np.std(envhum, axis=(1,2)) / means


    filename = Path(inputs_path) / "dhw" / (str(survey_id) + '.npy')
    dhw = np.load(filename).astype(np.float32)
    dhw_vect = np.array([np.mean(dhw), np.std(dhw) / np.mean(dhw), np.max(dhw)])

    return np.concatenate([center_values, means, deviations, dhw_vect])

# X = fulldf[['subset']].progress_apply(get_compound_values, axis=1, result_type='expand')
# X.to_csv(root / f'X_compound_values_mm.csv')

In [3]:
## Prepare datasets

X = pd.read_csv(root / f'X_compound_values_mm.csv', index_col = 0)

def get_datasets(sp, modeltype='rf', objective='pa'):

    # X
    X_train = X.loc[fulldf['subset'] == 'train']
    X_val = X.loc[fulldf['subset'] == 'val']
    X_test = X.loc[fulldf['subset'] == 'test']

    # Y
    if objective == 'pa':
        targets = (groundtruth[sp] > 0).astype("category")
    else:
        targets = groundtruth[sp].astype(float)

    Y_train = targets.loc[fulldf['subset'] == 'train']
    Y_val = targets.loc[fulldf['subset'] == 'val']
    Y_test = targets.loc[fulldf['subset'] == 'test']

    if modeltype == 'xgb':

        dtrain_clf = xgb.DMatrix(X_train, Y_train, enable_categorical=True)
        dval_clf = xgb.DMatrix(X_val, Y_val, enable_categorical=True)
        dtest_clf = xgb.DMatrix(X_test, Y_test, enable_categorical=True)

        return dtrain_clf, dval_clf, dtest_clf
    
    else:

        return X_train, Y_train, X_val, Y_val, X_test, Y_test

## P-A

#### Train XGB

In [ ]:
for ss in np.arange(0.1, 1.0, 0.2):
   for sp in tqdm(species):

      dtrain_clf, dval_clf, dtest_clf = get_datasets(sp, 'xgb', 'pa')
      evals = [(dtrain_clf, "train"), (dval_clf, "validation")]

      params = {"objective": "binary:logistic", "tree_method": "hist", "device": "cuda","subsample":ss,"max_depth":4,"eval_metric":"auc","base_score":0.5}
      n=500

      model = xgb.train(
         params=params,
         dtrain=dtrain_clf,
         num_boost_round=n,
         evals=evals,
         verbose_eval=None,
         #early_stopping_rounds=10
      )

      targets = (groundtruth[sp] > 0).astype(float)
      Y_train = targets.loc[fulldf['subset'] == 'train'].astype('category')
      Y_val = targets.loc[fulldf['subset'] == 'val'].astype('category')
      Y_test = targets.loc[fulldf['subset'] == 'test'].astype('category')

      pd.Series(model.predict(dtrain_clf), name=sp, index = Y_train.index).to_csv(root / 'pa' / f'xgb-preds-{ss:.1f}' / f"train_{sp.replace('/','-')}.csv")
      pd.Series(model.predict(dval_clf), name=sp, index = Y_val.index).to_csv(root / 'pa' / f'xgb-preds-{ss:.1f}' / f"val_{sp.replace('/','-')}.csv")
      pd.Series(model.predict(dtest_clf), name=sp, index = Y_test.index).to_csv(root / 'pa' / f'xgb-preds-{ss:.1f}' / f"test_{sp.replace('/','-')}.csv")

#### Binarization and val F1

In [6]:
def rvalue(thres, df, target_sr):
    sr = (df > thres).astype(int).sum(axis=1)
    
    return np.abs(sr.mean() - target_sr.mean())


for ss in np.arange(0.1, 1.0, 0.2):

    # Calculate threshold

    xgbdict = {}

    for sp in tqdm(species):

        xgbdict[sp] = pd.read_csv(root / 'pa' / f'xgb-preds-{ss:.1f}' / f"val_{sp.replace('/','-')}.csv", index_col=0).squeeze()

    xgb_val = pd.DataFrame(xgbdict)

    targets_sr = (fulldf.loc[xgb_val.index, xgb_val.columns] > 0).sum(axis=1)
    xgb_threshold = minimize_scalar(rvalue, args=(xgb_val, targets_sr),method='Bounded', bounds=(0,1))['x']

    # Calculate val F1-scores

    xgb_val_pa = (pd.DataFrame(xgbdict) > xgb_threshold)

    for s in species:

        targ = (fulldf.loc[xgb_val_pa.index, s] > 0).astype(int).to_numpy().flatten()
        xgb_preds = xgb_val_pa[s].astype(int).to_numpy().flatten()
        xgbdict[s] = {'f1': f1_score(targ, xgb_preds, zero_division=0)}


    xgb_val_f1 = pd.DataFrame(xgbdict).T.sort_values('f1', ascending=False)
    xgb_val_f1.to_csv(root / 'pa' / f"xgb_val_f1-{ss:.1f}-{xgb_threshold:.3f}.csv")

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

#### Calculate F1 on test set

In [3]:
def best_f1(row):
    sp = row.name
    ss = row['ss']
    threshold = row['threshold']

    preds_test = pd.read_csv(root / 'pa' / f"xgb-preds-{ss}" / f"test_{sp.replace('/','-')}.csv", index_col=0).squeeze()
    preds_test_pa = (preds_test > threshold).astype(int).to_numpy().flatten()

    targ = (fulldf.loc[preds_test.index, sp] > 0).astype(int).to_numpy().flatten()

    return(f1_score(targ, preds_test_pa, zero_division=0))


In [4]:
scoresxgb = pd.DataFrame(index = species)
threshold = {}

for ss in np.arange(0.1, 1.0, 0.2):

    p = next(Path(root / 'pa').glob(f"xgb_val_f1-{ss:.1f}-*.csv"))
    scoresxgb[f"{ss:.1f}"] = pd.read_csv(p, index_col = 0)['f1']
    threshold[f"{ss:.1f}"] = float(p.stem.split('-')[-1])

best_idx = pd.DataFrame()
best_idx['ss'] = scoresxgb.idxmax(axis=1)
best_idx['threshold'] = best_idx['ss'].map(threshold)

f1 = best_idx.progress_apply(best_f1, axis=1)
xgb_f1 = pd.DataFrame(f1, columns=["f1"]).sort_values('f1', ascending=False)
xgb_f1.to_csv(root / 'pa' / f"xgbbest_f1--.4rank={len(xgb_f1[xgb_f1['f1']>=0.4])}.csv")

  0%|          | 0/817 [00:00<?, ?it/s]

#### Calculate F1 by site

In [81]:
scoresxgb = pd.DataFrame(index = species)
threshold = {}

for ss in np.arange(0.1, 1.0, 0.2):

    p = next(Path(root / 'pa').glob(f"xgb_val_f1-{ss:.1f}-*.csv"))
    scoresxgb[f"{ss:.1f}"] = pd.read_csv(p, index_col = 0)['f1']
    threshold[f"{ss:.1f}"] = float(p.stem.split('-')[-1])

best_idx = pd.DataFrame()
best_idx['ss'] = scoresxgb.idxmax(axis=1)
best_idx['threshold'] = best_idx['ss'].map(threshold)

In [ ]:
predictions = []
for sp in species:
    predictions.append(pd.read_csv(root / 'pa' / f"xgb-preds-{best_idx.loc[sp,'ss']}" / f"test_{sp.replace('/','-')}.csv", index_col=0).squeeze() >= best_idx.loc[sp,'threshold'])

predictions = pd.concat(predictions, names = species, axis = 1)
f1scores = []
test_df = fulldf[fulldf['subset'] == 'test']


for site in test_df['site_code'].unique():
    surveys = test_df[test_df['site_code'] == site].index
    lat, lon = test_df.loc[surveys[0], 'latitude'], test_df.loc[surveys[0], 'longitude']
    preds = predictions.astype(float).loc[surveys].to_numpy().flatten()
    targets = (test_df != 0).astype(float).loc[surveys, predictions.columns].to_numpy().flatten()
    f1 = f1_score(targets, preds)
    f1scores.append({'site':site, 'f1':f1, 'lat':lat, 'lon':lon})

pd.DataFrame(f1scores).to_csv(root / 'pa' / f"xgbbest_f1_by_site.csv")

## Reg

#### Train XGB

In [4]:
for ss in np.arange(0.1, 1.0, 0.2):
    for sp in tqdm(species):

      dtrain_clf, dval_clf, dtest_clf = get_datasets(sp, 'xgb', 'reg')
      evals = [(dtrain_clf, "train"), (dval_clf, "validation")]

      params = {"objective": "reg:squarederror", "tree_method": "hist", "device": "cuda","subsample":ss,"max_depth":4}
      n=500

      model = xgb.train(
        params=params,
        dtrain=dtrain_clf,
        num_boost_round=n,
        evals=evals,
        verbose_eval=None,
        early_stopping_rounds=10
      )

      targets = (groundtruth[sp] > 0).astype(float)
      Y_train = targets.loc[fulldf['subset'] == 'train'].astype('category')
      Y_val = targets.loc[fulldf['subset'] == 'val'].astype('category')
      Y_test = targets.loc[fulldf['subset'] == 'test'].astype('category')

      pd.Series(model.predict(dtrain_clf), name=sp, index = Y_train.index).to_csv(root / 'reg' / f'xgb-preds-{ss:.1f}' / f"train_{sp.replace('/','-')}.csv")
      pd.Series(model.predict(dval_clf), name=sp, index = Y_val.index).to_csv(root / 'reg' / f'xgb-preds-{ss:.1f}' / f"val_{sp.replace('/','-')}.csv")
      pd.Series(model.predict(dtest_clf), name=sp, index = Y_test.index).to_csv(root / 'reg' / f'xgb-preds-{ss:.1f}' / f"test_{sp.replace('/','-')}.csv")

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

#### Calculate R2 on val set

In [ ]:
for ss in np.arange(0.1, 1.0, 0.2):
    dic = {}
    for s in tqdm(species):

        predsxgb = pd.read_csv(root / 'reg' / f"xgb-preds-{ss:.1f}" / f"val_{s.replace('/','-')}.csv", index_col=0).squeeze()
        
        dic[s] = {'pearsonr': pearsonr(groundtruth.loc[predsxgb.index, s], np.exp(predsxgb)-1)[0]}
        
    scoresxgb = pd.DataFrame(dic).T.sort_values(ascending=False, by='pearsonr')
    scoresxgb.to_csv(root / 'reg' / f"xgb_val_r2-{ss:.1f}.csv")

#### Calculate test R2

In [7]:
def best_r2(row):
    sp = row.name
    ss = row['ss']

    preds_test = pd.read_csv(root / 'reg' / f"xgb-preds-{ss}" / f"test_{sp.replace('/','-')}.csv", index_col=0).squeeze()
    targ = groundtruth.loc[preds_test.index, sp]

    nonzero = (targ > 0) * (preds_test > 0)

    res = {'pearsonr': pearsonr(targ, preds_test)[0],
            'spearmanr': spearmanr(targ, preds_test)[0]}
    
    if nonzero.sum() > 1:
        res['nzpearsonr'] =  pearsonr(targ.loc[nonzero], preds_test.loc[nonzero])[0]
        res['nzspearmanr'] =  spearmanr(targ.loc[nonzero], preds_test.loc[nonzero])[0]

    return res

In [15]:
r2s


,0,1
Abudefduf bengalensis,0.296372,0.189291
Abudefduf sexfasciatus,0.280823,0.284623
Abudefduf vaigiensis,0.127699,0.169203
Abudefduf whitleyi,0.253996,0.142197
Acanthaluteres brownii,-0.003548,0.019661
...,...,...
Variola louti,0.026240,0.080772
Vincentia conspersa,0.079024,0.033840
Zanclus cornutus,0.510456,0.343123
Zebrasoma scopas,0.452681,0.445866


In [9]:
scoresxgb = pd.DataFrame(index = species)

for ss in np.arange(0.1, 1.0, 0.2):

    scoresxgb[f"{ss:.1f}"] = pd.read_csv(root / 'reg' / f"xgb_val_r2-{ss:.1f}.csv", index_col = 0)['pearsonr']

best_idx = pd.DataFrame()
best_idx['ss'] = scoresxgb.idxmax(axis=1).fillna('0.1')

r2s = best_idx.progress_apply(best_r2, axis=1, result_type='expand')
r2s.columns = ['pearsonr', 'spearmanr', 'pearsonr_nz', 'spearmanr_nz']
xgb_r2s = r2s.sort_values('pearsonr', ascending=False).fillna(-.1)
xgb_r2s.to_csv(root / 'reg' / f"xgb_best_r2--.4rank={len(xgb_r2s[xgb_r2s['pearsonr']>=0.4])}.csv")

/tmp/ipykernel_21829/2533499809.py:8: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  best_idx['ss'] = scoresxgb.idxmax(axis=1).fillna('0.1')


  0%|          | 0/817 [00:00<?, ?it/s]

/tmp/ipykernel_21829/1686655875.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  res['nzpearsonr'] =  pearsonr(targ.loc[nonzero], preds_test.loc[nonzero])[0]
/tmp/ipykernel_21829/1686655875.py:15: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  res['nzspearmanr'] =  spearmanr(targ.loc[nonzero], preds_test.loc[nonzero])[0]
/tmp/ipykernel_21829/1686655875.py:10: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  res = {'pearsonr': pearsonr(targ, preds_test)[0],
/tmp/ipykernel_21829/1686655875.py:11: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  'spearmanr': spearmanr(targ, preds_test)[0]}
/tmp/ipykernel_21829/1686655875.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  res['nzpearsonr'] =  pearsonr(targ.loc[nonzero], preds_test.loc[nonzero])[

## Eco indicators

#### Train XGB

In [4]:
## Prepare datasets
eco_gt = pd.read_csv(inputs_path / 'eco_indic_norm.csv', index_col='survey_id')
species = list(eco_gt.columns)[1:-1]
targets = np.log(1+eco_gt[species])

def get_datasets(sp):

    # X
    X = pd.read_csv(root / f'X_compound_values_mm.csv', index_col = 0)
    X_train = X.loc[eco_gt[eco_gt['subset'] == 'train'].index]
    X_val = X.loc[eco_gt[eco_gt['subset'] == 'val'].index]
    X_test = X.loc[eco_gt[eco_gt['subset'] == 'test'].index]

    # Y
    Y_train = targets.loc[eco_gt['subset'] == 'train', sp]
    Y_val = targets.loc[eco_gt['subset'] == 'val', sp]
    Y_test = targets.loc[eco_gt['subset'] == 'test', sp]

    dtrain_clf = xgb.DMatrix(X_train, Y_train)
    dval_clf = xgb.DMatrix(X_val, Y_val)
    dtest_clf = xgb.DMatrix(X_test, Y_test)

    return dtrain_clf, dval_clf, dtest_clf


In [5]:
for ss in np.arange(0.1, 1.0, 0.2):
    for sp in tqdm(species):

      dtrain_clf, dval_clf, dtest_clf = get_datasets(sp)
      evals = [(dtrain_clf, "train"), (dval_clf, "validation")]

      params = {"objective": "reg:squarederror", "tree_method": "hist", "device": "cuda","subsample":ss,"max_depth":4}
      n=200

      model = xgb.train(
        params=params,
        dtrain=dtrain_clf,
        num_boost_round=n,
        evals=evals,
        verbose_eval=5,
        early_stopping_rounds=10
      )

      Y_train = eco_gt.loc[eco_gt['subset'] == 'train']
      Y_val = eco_gt.loc[eco_gt['subset'] == 'val']
      Y_test = eco_gt.loc[eco_gt['subset'] == 'test']

      pd.Series(model.predict(dtrain_clf), name=sp, index = Y_train.index).to_csv(root / 'eco' / f'xgb-preds-{ss:.1f}' / f"train_{sp.replace('/','-')}.csv")
      pd.Series(model.predict(dval_clf), name=sp, index = Y_val.index).to_csv(root / 'eco' / f'xgb-preds-{ss:.1f}' / f"val_{sp.replace('/','-')}.csv")
      pd.Series(model.predict(dtest_clf), name=sp, index = Y_test.index).to_csv(root / 'eco' / f'xgb-preds-{ss:.1f}' / f"test_{sp.replace('/','-')}.csv")

  0%|          | 0/3 [00:00<?, ?it/s]

[0]	train-rmse:0.09568	validation-rmse:0.08807
[5]	train-rmse:0.05993	validation-rmse:0.06507
[10]	train-rmse:0.05525	validation-rmse:0.06175
[15]	train-rmse:0.05389	validation-rmse:0.06184
[20]	train-rmse:0.05262	validation-rmse:0.06310
[0]	train-rmse:0.06135	validation-rmse:0.05591
[5]	train-rmse:0.05055	validation-rmse:0.05049
[10]	train-rmse:0.04847	validation-rmse:0.04950
[15]	train-rmse:0.04772	validation-rmse:0.05028
[20]	train-rmse:0.04683	validation-rmse:0.05186
[0]	train-rmse:0.16599	validation-rmse:0.18120
[5]	train-rmse:0.14629	validation-rmse:0.16600
[10]	train-rmse:0.14242	validation-rmse:0.16235
[15]	train-rmse:0.14111	validation-rmse:0.16210
[20]	train-rmse:0.14065	validation-rmse:0.16343
[21]	train-rmse:0.14055	validation-rmse:0.16352


  0%|          | 0/3 [00:00<?, ?it/s]

[0]	train-rmse:0.09469	validation-rmse:0.08651
[5]	train-rmse:0.05915	validation-rmse:0.06359
[10]	train-rmse:0.05384	validation-rmse:0.06090
[15]	train-rmse:0.05180	validation-rmse:0.06093
[20]	train-rmse:0.05041	validation-rmse:0.06083
[25]	train-rmse:0.04940	validation-rmse:0.06138
[29]	train-rmse:0.04849	validation-rmse:0.06178
[0]	train-rmse:0.06121	validation-rmse:0.05551
[5]	train-rmse:0.04922	validation-rmse:0.05088
[10]	train-rmse:0.04623	validation-rmse:0.05100
[15]	train-rmse:0.04495	validation-rmse:0.05135
[17]	train-rmse:0.04441	validation-rmse:0.05187
[0]	train-rmse:0.16601	validation-rmse:0.18379
[5]	train-rmse:0.14473	validation-rmse:0.16873
[10]	train-rmse:0.13935	validation-rmse:0.16424
[15]	train-rmse:0.13640	validation-rmse:0.16403
[20]	train-rmse:0.13499	validation-rmse:0.16385
[25]	train-rmse:0.13329	validation-rmse:0.16436
[28]	train-rmse:0.13227	validation-rmse:0.16521


  0%|          | 0/3 [00:00<?, ?it/s]

[0]	train-rmse:0.09461	validation-rmse:0.08573
[5]	train-rmse:0.05885	validation-rmse:0.06282
[10]	train-rmse:0.05372	validation-rmse:0.05944
[15]	train-rmse:0.05136	validation-rmse:0.05923
[20]	train-rmse:0.05008	validation-rmse:0.05905
[21]	train-rmse:0.04982	validation-rmse:0.05908
[0]	train-rmse:0.06098	validation-rmse:0.05630
[5]	train-rmse:0.04905	validation-rmse:0.05006
[10]	train-rmse:0.04607	validation-rmse:0.04926
[15]	train-rmse:0.04428	validation-rmse:0.04948
[20]	train-rmse:0.04331	validation-rmse:0.04958
[21]	train-rmse:0.04306	validation-rmse:0.04975
[0]	train-rmse:0.16632	validation-rmse:0.18240
[5]	train-rmse:0.14330	validation-rmse:0.17064
[10]	train-rmse:0.13817	validation-rmse:0.16791
[15]	train-rmse:0.13590	validation-rmse:0.16755
[20]	train-rmse:0.13373	validation-rmse:0.16781
[25]	train-rmse:0.13229	validation-rmse:0.16770
[28]	train-rmse:0.13129	validation-rmse:0.16956


  0%|          | 0/3 [00:00<?, ?it/s]

[0]	train-rmse:0.09468	validation-rmse:0.08611
[5]	train-rmse:0.05836	validation-rmse:0.06558
[10]	train-rmse:0.05299	validation-rmse:0.06360
[15]	train-rmse:0.05098	validation-rmse:0.06211
[20]	train-rmse:0.04975	validation-rmse:0.06161
[25]	train-rmse:0.04872	validation-rmse:0.06154
[30]	train-rmse:0.04769	validation-rmse:0.06172
[31]	train-rmse:0.04748	validation-rmse:0.06179
[0]	train-rmse:0.06101	validation-rmse:0.05565
[5]	train-rmse:0.04846	validation-rmse:0.05004
[10]	train-rmse:0.04546	validation-rmse:0.05092
[15]	train-rmse:0.04391	validation-rmse:0.05085
[0]	train-rmse:0.16605	validation-rmse:0.18103
[5]	train-rmse:0.14190	validation-rmse:0.16601
[10]	train-rmse:0.13819	validation-rmse:0.16617
[15]	train-rmse:0.13572	validation-rmse:0.16539
[17]	train-rmse:0.13506	validation-rmse:0.16476


  0%|          | 0/3 [00:00<?, ?it/s]

[0]	train-rmse:0.09461	validation-rmse:0.08600
[5]	train-rmse:0.05833	validation-rmse:0.06385
[10]	train-rmse:0.05303	validation-rmse:0.06080
[15]	train-rmse:0.05106	validation-rmse:0.06056
[20]	train-rmse:0.04976	validation-rmse:0.06087
[25]	train-rmse:0.04877	validation-rmse:0.06095
[0]	train-rmse:0.06112	validation-rmse:0.05617
[5]	train-rmse:0.04860	validation-rmse:0.05060
[10]	train-rmse:0.04557	validation-rmse:0.05051
[15]	train-rmse:0.04397	validation-rmse:0.05051
[18]	train-rmse:0.04343	validation-rmse:0.05071
[0]	train-rmse:0.16593	validation-rmse:0.17984
[5]	train-rmse:0.14210	validation-rmse:0.16766
[10]	train-rmse:0.13766	validation-rmse:0.16382
[15]	train-rmse:0.13534	validation-rmse:0.16361
[20]	train-rmse:0.13305	validation-rmse:0.16461
[25]	train-rmse:0.13100	validation-rmse:0.16815
[27]	train-rmse:0.13033	validation-rmse:0.16818


#### Calculate R2 on val set

In [6]:
for ss in np.arange(0.1, 1.0, 0.2):
    dic = {}
    for s in tqdm(species):

        predsxgb = pd.read_csv(root / 'eco' / f"xgb-preds-{ss:.1f}" / f"val_{s.replace('/','-')}.csv", index_col=0).squeeze()
        
        dic[s] = {'pearsonr': pearsonr(targets.loc[predsxgb.index, s], predsxgb)[0]}
        
    scoresxgb = pd.DataFrame(dic).T.sort_values(ascending=False, by='pearsonr')
    scoresxgb.to_csv(root / 'eco' / f"xgb_val_r2-{ss:.1f}.csv")

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

#### Calculate test R2

In [7]:
def best_r2(row):
    sp = row.name
    ss = row['ss']

    preds_test = pd.read_csv(root / 'eco' / f"xgb-preds-{ss}" / f"test_{sp.replace('/','-')}.csv", index_col=0).squeeze()
    targ = eco_gt.loc[preds_test.index, sp]

    return(pearsonr(targ, preds_test)[0])

In [8]:
scoresxgb = pd.DataFrame(index = species)

for ss in np.arange(0.1, 1.0, 0.2):

    scoresxgb[f"{ss:.1f}"] = pd.read_csv(root / 'eco' / f"xgb_val_r2-{ss:.1f}.csv", index_col = 0)['pearsonr']

best_idx = pd.DataFrame()
best_idx['ss'] = scoresxgb.idxmax(axis=1).fillna('0.1')

r2s = best_idx.progress_apply(best_r2, axis=1)
xgb_r2s = pd.DataFrame(r2s, columns=["pearsonr"]).sort_values('pearsonr', ascending=False).fillna(-.1)
xgb_r2s.to_csv(root / 'eco' / f"xgb_best_r2--.4rank={len(xgb_r2s[xgb_r2s['pearsonr']>=0.4])}.csv")

  0%|          | 0/3 [00:00<?, ?it/s]

## Old code: RF

In [ ]:
# PA
 
for sp in tqdm(species):

    X_train, Y_train, X_val, Y_val, X_test, Y_test = get_datasets(sp, 'rf', 'pa')
    regr = RandomForestClassifier(min_samples_split=2, # Useless
                                    min_samples_leaf=3,
                                    max_leaf_nodes=None,
                                    n_estimators=300,
                                    max_depth=None,
                                    random_state=0,
                                    n_jobs=32)

    # Export predictions

    regr.fit(pd.concat([X_train, X_val]), pd.concat([Y_train, Y_val]))

    # Export predictions
    if regr.n_classes_ == 2:
        pd.Series(regr.predict_proba(X_train)[:, 1], name=sp, index = Y_train.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"train_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict_proba(X_val)[:, 1], name=sp, index = Y_val.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"val_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict_proba(X_test)[:, 1], name=sp, index = Y_test.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"test_{sp.replace('/','-')}.csv")
    else:
        pd.Series([0] * len(Y_train), name=sp, index = Y_train.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"train_{sp.replace('/','-')}.csv")
        pd.Series([0] * len(Y_val), name=sp, index = Y_val.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"val_{sp.replace('/','-')}.csv")
        pd.Series([0] * len(Y_test), name=sp, index = Y_test.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"test_{sp.replace('/','-')}.csv")

In [ ]:
# Bins 

for i in [5, 10, 20]:
    for sp in tqdm(species):

        X_train, Y_train, X_val, Y_val, X_test, Y_test = get_datasets(sp, 'rf', 'pa', i)
        regr = RandomForestClassifier(min_samples_split=2, # Useless
                                        min_samples_leaf=3,
                                        max_leaf_nodes=None,
                                        n_estimators=300,
                                        max_depth=None,
                                        random_state=0,
                                        n_jobs=32)

        # Export predictions

        regr.fit(pd.concat([X_train, X_val]), pd.concat([Y_train, Y_val]))

        # Export predictions
        pd.Series(regr.predict(X_train), name=sp, index = Y_train.index).to_csv(root / 'bins' / f"rf{i}-preds-tv" / f"train_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict(X_val), name=sp, index = Y_val.index).to_csv(root / 'bins' / f"rf{i}-preds-tv" / f"val_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict(X_test), name=sp, index = Y_test.index).to_csv(root / 'bins' / f"rf{i}-preds-tv" / f"test_{sp.replace('/','-')}.csv")


for i in [5, 10, 20]:
    for sp in tqdm(species):

        X_train, Y_train, X_val, Y_val, X_test, Y_test = get_datasets(sp, 'rf', 'pa', i)
        regr = RandomForestClassifier(min_samples_split=2, # Useless
                                        min_samples_leaf=3,
                                        max_leaf_nodes=None,
                                        n_estimators=300,
                                        max_depth=None,
                                        random_state=0,
                                        n_jobs=32)

        # Export predictions

        regr.fit(X_train, Y_train)

        # Export predictions
        pd.Series(regr.predict(X_train), name=sp, index = Y_train.index).to_csv(root / 'bins' / f"rf{i}-preds" / f"train_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict(X_val), name=sp, index = Y_val.index).to_csv(root / 'bins' / f"rf{i}-preds" / f"val_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict(X_test), name=sp, index = Y_test.index).to_csv(root / 'bins' / f"rf{i}-preds" / f"test_{sp.replace('/','-')}.csv")

In [ ]:
# Reg

for sp in tqdm(species):

    X_train, Y_train, X_val, Y_val, X_test, Y_test = get_datasets(sp, 'rf', 'reg')
    regr = RandomForestRegressor(min_samples_split=2, # Useless
                                    min_samples_leaf=3,
                                    max_leaf_nodes=None,
                                    n_estimators=300,
                                    max_depth=None,
                                    random_state=0,
                                    n_jobs=32)

    # Export predictions

    regr.fit(X_train, Y_train)

    # Export predictions
    pd.Series(regr.predict(X_train), name=sp, index = Y_train.index).to_csv(root / 'reg' / f"rf-preds" / f"train_{sp.replace('/','-')}.csv")
    pd.Series(regr.predict(X_val), name=sp, index = Y_val.index).to_csv(root / 'reg' / f"rf-preds" / f"val_{sp.replace('/','-')}.csv")
    pd.Series(regr.predict(X_test), name=sp, index = Y_test.index).to_csv(root / 'reg' / f"rf-preds" / f"test_{sp.replace('/','-')}.csv")

    X_train, Y_train, X_val, Y_val, X_test, Y_test = get_datasets(sp, 'rf', 'reg')
    regr = RandomForestRegressor(min_samples_split=2, # Useless
                                    min_samples_leaf=3,
                                    max_leaf_nodes=None,
                                    n_estimators=300,
                                    max_depth=None,
                                    random_state=0,
                                    n_jobs=32)

    # Export predictions

    regr.fit(pd.concat([X_train, X_val]), pd.concat([Y_train, Y_val]))

    # Export predictions
    pd.Series(regr.predict(X_train), name=sp, index = Y_train.index).to_csv(root / 'reg' / f"rf-preds-tv" / f"train_{sp.replace('/','-')}.csv")
    pd.Series(regr.predict(X_val), name=sp, index = Y_val.index).to_csv(root / 'reg' / f"rf-preds-tv" / f"val_{sp.replace('/','-')}.csv")
    pd.Series(regr.predict(X_test), name=sp, index = Y_test.index).to_csv(root / 'reg' / f"rf-preds-tv" / f"test_{sp.replace('/','-')}.csv")

## Not used: Bins

In [ ]:
## Bins data
groundtruth_bins = {}
for i in [5, 10, 20]:
    df = pd.read_csv(inputs_path / f"database_common_{i}binned.csv", index_col='survey_id',
                     dtype = {22:str, 24:str, 25:str})
    groundtruth_bins[i] = df[species]

#### Train XGB

In [ ]:
i = 10
for ss in np.arange(0.1, 1.0, 0.2):
    for sp in tqdm(species):

        dtrain_clf, dval_clf, dtest_clf = get_datasets(sp, 'xgb', 'bins', i)
        evals = [(dtrain_clf, "train"), (dval_clf, "validation")]

        params = {"objective": "multi:softmax", "tree_method": "hist", "device": "cuda","subsample":ss,"max_depth":4,"num_class":i}
        n=100

        model = xgb.train(
            params=params,
            dtrain=dtrain_clf,
            num_boost_round=n,
            evals=evals,
            verbose_eval=None
        )

        targets = (groundtruth[sp] > 0).astype(float)
        Y_train = targets.loc[fulldf['subset'] == 'train'].astype('category')
        Y_val = targets.loc[fulldf['subset'] == 'val'].astype('category')
        Y_test = targets.loc[fulldf['subset'] == 'test'].astype('category')

        pd.Series(model.predict(dtrain_clf), name=sp, index = Y_train.index).to_csv(root / 'bins' / f"xgb{i}-preds-{ss:.1f}" / f"train_{sp.replace('/','-')}.csv")
        pd.Series(model.predict(dval_clf), name=sp, index = Y_val.index).to_csv(root / 'bins' / f"xgb{i}-preds-{ss:.1f}" / f"val_{sp.replace('/','-')}.csv")
        pd.Series(model.predict(dtest_clf), name=sp, index = Y_test.index).to_csv(root / 'bins' / f"xgb{i}-preds-{ss:.1f}" / f"test_{sp.replace('/','-')}.csv")

#### Calculate R2 on val set

In [ ]:
## val

i = 10
earlystopping = '-es'

for ss in np.arange(0.1, 1.0, 0.2):

    # Calculate threshold

    xgbdict = {}

    for sp in tqdm(species):

        xgb_val = pd.read_csv(root / 'bins' / f'xgb{i}-preds-{ss:.1f}{earlystopping}' / f"val_{sp.replace('/','-')}.csv", index_col=0).squeeze()
        targ = groundtruth_bins[i].loc[xgb_val.index, sp]
        xgbdict[sp] = {'pearsonr': pearsonr(groundtruth_bins[i].loc[xgb_val.index, sp], xgb_val)[0]}


    xgb_val_f1 = pd.DataFrame(xgbdict).T.sort_values('pearsonr', ascending=False)
    xgb_val_f1.to_csv(root / 'bins' / f"xgb{i}_val_r2-{ss:.1f}{earlystopping}.csv")

#### Calculate test R2

In [ ]:
i = 10

with open(inputs_path / f"database_common_{i}bins.txt", "r") as f:
        medians = f.readlines()
        medians = [float(m.strip()) for m in medians]

        median_dic = {i: medians[i] for i in range(len(medians))}


def best_r2(row):
    sp = row.name
    ss = row['ss']

    preds_test = pd.read_csv(root / 'bins' / f"xgb{i}-preds-{ss}" / f"test_{sp.replace('/','-')}.csv", index_col=0).squeeze()
    targ = groundtruth_bins[i].loc[preds_test.index, sp]

    median_preds_xgb = preds_test.astype(int).replace(median_dic)
    median_pearson = pearsonr(groundtruth.loc[preds_test.index, sp], median_preds_xgb)[0]

    return(pearsonr(targ, preds_test)[0], median_pearson)


In [ ]:
scoresxgb = pd.DataFrame(index = species)

for ss in np.arange(0.1, 1.0, 0.2):

    scoresxgb[f"{ss:.1f}"] = pd.read_csv(root / 'bins' / f"xgb{i}_val_r2-{ss:.1f}.csv", index_col = 0)['pearsonr']

best_idx = pd.DataFrame()
best_idx['ss'] = scoresxgb.idxmax(axis=1).fillna('0.1')


r2s = best_idx.progress_apply(best_r2, axis=1, result_type='expand')
r2s.columns = ['pearsonr', 'pearsonr_median']
xgb_r2s = r2s.sort_values('pearsonr', ascending=False).fillna(-.1)
xgb_r2s.to_csv(root / 'bins' / f"xgb{i}_best_r2--.4rank={len(xgb_r2s[xgb_r2s['pearsonr']>=0.4])}.csv")